In [1]:
on_colab = False

import os

if not on_colab:
    root_folder = '../..'

store_attn_on_local: bool = False
store_rel_on_local:  bool = True

if store_attn_on_local or store_rel_on_local:
    if on_colab:
        if os.path.exists('/src/TCT-visual-search/results'):
            os.unlink('/src/TCT-visual-search/results')
        os.symlink(os.path.abspath('../results'), '/src/TCT-visual-search/results')

# _________ APPEND SOME PATH TO IMPORT SOME LIBRARY BELOW _____________
import sys

sys.path.append('..')
sys.path.append('../ViT_utils')
# _________ APPEND SOME PATH TO IMPORT SOME LIBRARY BELOW _____________

In [2]:
import sys
import cv2
import time
from matplotlib import pyplot as plt
from tqdm import tqdm, trange
import numpy as np
import pandas as pd
import pickle
import random
import copy

import os
import shutil
from PIL import Image, ImageDraw


import torch
from torch.utils.data import Dataset
from torchvision.transforms.functional import to_tensor, normalize
from torchvision import transforms

import matplotlib.pyplot as plt
import numpy as np
from scipy.datasets import face
from scipy.ndimage import zoom
from scipy.special import logsumexp
import torch
import deepgaze_pytorch
from deepgaze_pytorch import modules

from naturaldesign.naturaldesign import NaturalDesign
from utils import *
sys.path.append("..")

In [3]:
test_imagedir = "../datasets/naturaldesign/naturaldesign/"
dataset = NaturalDesign(test_imagedir)

In [4]:
def logsearchProcess(x, y, tg_xy, attentionMap, image_size, size, coef):
    mask_size = size
    tg_x, tg_y, w, h = tg_xy
    tg_xmax, tg_ymax = tg_x + w, tg_y + h 

    attenNP = (attentionMap[0,:,:].detach() * coef[0,:,:].detach()).numpy()
    y_fix, x_fix = y, x

    x_max_s, x_min_s, y_max_s, y_min_s = min(x_fix+mask_size//2, image_size[1]-1), max(x_fix-mask_size//2, 0), min(y_fix+mask_size//2, image_size[0]-1), max(y_fix-mask_size//2, 0)

    if x_max_s < tg_x or x_min_s > tg_xmax or y_max_s < tg_y or y_min_s > tg_ymax:
        coef[0, y_min_s:y_max_s+1, x_min_s:x_max_s+1] = 1000
        attenNP = (attentionMap[0,:,:].detach() * coef[0,:,:].detach()).numpy()
        y_fix, x_fix = np.unravel_index(attenNP.argmax(), attenNP.shape)
        return False, (x_fix, y_fix), coef

    return True, [], coef

def fixation_initialize():
    k, x_range, y_range = 4, 80, 65
    x_init, y_init = 640//2, 512//2
    random_num_x, random_num_y = random.sample(list(range(x_range)), 4), random.sample(list(range(y_range)), 4)
    x, y = [], []
    for i in range(4):
        if i < 3:
            choice_x, choice_y = random.choice([0, 1]), random.choice([0, 1])
            x_cord = int(x_init+random_num_x[i]) if choice_x == 0 else int(x_init-random_num_x[i])
            y_cord = int(y_init+random_num_y[i]) if choice_y == 0 else int(y_init-random_num_y[i])
            x.append(x_cord)
            y.append(y_cord)
        else:
            x.append(320)
            y.append(256)

    return x, y

In [5]:
fixation_initialize()

([326, 304, 361, 320], [248, 195, 202, 256])

In [6]:

DEVICE = 'mps'
img_size = (512, 640)
# you can use DeepGazeI or DeepGazeIIE
model = deepgaze_pytorch.DeepGazeIII(pretrained=True).to(DEVICE)
image = face()
centerbias_template = np.load('centerbias_mit1003.npy')
# rescale to match image size
centerbias = zoom(centerbias_template, (image.shape[0]/centerbias_template.shape[0], image.shape[1]/centerbias_template.shape[1]), order=0, mode='nearest')
# renormalize log density
centerbias -= logsumexp(centerbias)
centerbias_tensor = torch.tensor([centerbias], dtype=torch.float32).to(DEVICE)
centerbias_tensor = transforms.Resize(img_size)(centerbias_tensor)

Using cache found in /Users/nguyentuan/.cache/torch/hub/pytorch_vision_v0.6.0
/Users/nguyentuan/Mirror/python_env/FYP_env/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nguyentuan/Mirror/python_env/FYP_env/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/var/folders/ln/xzfjqkm15md4971bjrhc8szr0000gn/T/ipykernel_54912/1582620585.py:11: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray wi

In [8]:
deepgaze3_res, size = [], 100
scanpath = {}

for i in trange(1, len(dataset)):
    img, bbox_relative = dataset[i]
    # get the target bounding box
    tg_loc = bbox_cordinates(bbox_relative, img_size[1], img_size[0])
    history_x, history_y = fixation_initialize()
    # transform img to tensor
    img_tensor = torch.tensor([img.transpose(2, 0, 1)]).to(DEVICE)
    img_tensor = transforms.Resize(img_size)(img_tensor)

    count, max_search, coef, path = 0, 999, torch.ones((1, 512, 640)), []
    while count < max_search:
        fixation_history_x = np.array(history_x)
        fixation_history_y = np.array(history_y)
        x_hist_tensor = torch.tensor([fixation_history_x[model.included_fixations]]).to(DEVICE)
        y_hist_tensor = torch.tensor([fixation_history_y[model.included_fixations]]).to(DEVICE)
        log_density_prediction = model(img_tensor, centerbias_tensor, x_hist_tensor, y_hist_tensor)

        path.append([history_x[-1], history_y[-1]])
        isTg, coordinates, coef = logsearchProcess(history_x[-1], history_y[-1], tg_loc, log_density_prediction.squeeze(0).cpu(), img_size, size, coef)
        count += 1

        if isTg: 
            break

        history_x.append(coordinates[0])
        history_y.append(coordinates[1])

    scanpath[i] = path
    print(count)
    deepgaze3_res.append(count)

  0%|                                                                                                     | 0/240 [00:00<?, ?it/s]/Users/nguyentuan/Mirror/python_env/FYP_env/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
  0%|▍                                                                                          | 1/240 [00:17<1:08:33, 17.21s/it]

40


  1%|▊                                                                                          | 2/240 [00:44<1:31:52, 23.16s/it]

78


  1%|█▏                                                                                         | 3/240 [00:55<1:08:54, 17.45s/it]

...

9


100%|██████████████████████████████████████████████████████████████████████████████████████████▌| 239/240 [26:36<00:07,  7.79s/it]

12


100%|███████████████████████████████████████████████████████████████████████████████████████████| 240/240 [26:40<00:00,  6.67s/it]

10


In [12]:
scanpath[1]

[[320, 256],
 [np.int64(371), np.int64(248)],
 [np.int64(422), np.int64(244)],
 [np.int64(473), np.int64(256)],
 [np.int64(524), np.int64(268)],
 [np.int64(472), np.int64(307)],
 [np.int64(460), np.int64(358)],
 [np.int64(511), np.int64(372)],
 [np.int64(523), np.int64(320)],
 [np.int64(296), np.int64(307)],
 [np.int64(244), np.int64(296)],
 [np.int64(184), np.int64(292)],
 [np.int64(132), np.int64(288)],
 [np.int64(347), np.int64(307)],
 [np.int64(420), np.int64(304)],
 [np.int64(460), np.int64(409)],
 [np.int64(511), np.int64(423)],
 [np.int64(408), np.int64(412)],
 [np.int64(352), np.int64(400)],
 [np.int64(300), np.int64(376)],
 [np.int64(248), np.int64(358)],
 [np.int64(196), np.int64(348)],
 [np.int64(144), np.int64(348)],
 [np.int64(128), np.int64(399)],
 [np.int64(179), np.int64(399)],
 [np.int64(230), np.int64(409)],
 [np.int64(284), np.int64(427)],
 [np.int64(336), np.int64(451)],
 [np.int64(392), np.int64(463)],
 [np.int64(443), np.int64(463)],
 [np.int64(494), np.int64(474)

In [13]:
deepgaze_naturaldesign_res = {}
deepgaze_naturaldesign_res['scanpath'] = scanpath

In [14]:
with open("../results/NaturalDesign/[Fig5]NaturalDesign_deepgaze_res.pkl", "wb") as tf:
    pickle.dump(deepgaze_naturaldesign_res, tf)

In [15]:
deepgaze3_accu_ND = [0] + model_performance(deepgaze3_res, len(deepgaze3_res))
deepgaze3_accu_ND[:11]

[0,
 np.float64(0.11666666666666667),
 np.float64(0.15416666666666667),
 np.float64(0.22083333333333333),
 np.float64(0.25833333333333336),
 np.float64(0.2875),
 np.float64(0.32083333333333336),
 np.float64(0.35),
 np.float64(0.39166666666666666),
 np.float64(0.4375),
 np.float64(0.45)]

In [16]:
with open("../results/naturaldesign_deepgaze3_accu_performance.pkl", "wb") as tf:
    pickle.dump(deepgaze3_accu_ND, tf)